# 04 — LEAR + Demir 2019 technical indicators (cross-regime)

Fourth milestone of sub-objective 3.1. Builds on notebook 03 by
asking whether Demir-style technical indicators (EMA, Bollinger %B,
MACD, MOM, ROC, Coppock) provide measurable predictive content on
MIBEL DAM when bolted onto the LEAR backbone, across the same three
regimes (2022 H2, 2023, 2024) plus the pooled 2023-2024 view.

**Parameters used.** All TI hyperparameters come from the audit at
`reports/diagnostics/demir_2019_ti_parameter_audit_2026_05.md`
(commits `0a81f1f` + `c52c3ed`). No grid-search on MIBEL is run in
this notebook; that ablation is listed there as future work.

**Five LEAR variants** compared per regime, alongside the seasonal naive:

1. **LEAR ar-only** — 103 features (price lags + DoW only); baseline.
2. **LEAR ar-only + TI** — 295 features (adds 8 × 24 = 192 TI cells).
3. **LEAR demand+wind** — 247 features (Lago 2021 canonical).
4. **LEAR demand+wind + TI** — 439 features.
5. **LEAR demand+solar+wind + TI** — 511 features.

**Pairwise comparisons reported** (DM test, MSE-loss differential,
Newey-West HAC with Andrews 1991 lag floored at `h-1` for `h=24`):

- `LEAR ar-only` vs `LEAR ar-only + TI` (isolates TI contribution
  without exogenous);
- `LEAR demand+wind` vs `LEAR demand+wind + TI` (isolates TI
  contribution on top of Lago's canonical configuration).

**Architecture.** Same as notebook 03: one pure `run_regime(spec)`
function in `src/mibel_forecasting/evaluation/_robustness.py`,
dispatched through `concurrent.futures.ProcessPoolExecutor` with at
most three workers. The TI columns are computed once (on the full
panel) inside `run_regime` itself via the `with_ti=True` spec flag
and joined to the panel before the per-model backtests.

**Validity preconditions.** Every result here inherits the UTC
convention (commit `b38b456`), the target masking (commit `00a0aaf`),
the partial-day uniform skipping (commits `89cd2db` + `3d3a912`), and
the LassoLarsIC noise-variance buffer (commit `685290b`). The TI
block adds its own audit-pinned shift(1) leakage-safety
(commit `4cccb0e`).


In [ ]:
from __future__ import annotations
import os
import time
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd

from mibel_forecasting.data.loaders import load_dam_panel
from mibel_forecasting.evaluation._robustness import (
    MODEL_NAMES_WITH_TI,
    cap_blas_threads,
    run_regime,
)
from mibel_forecasting.evaluation.dm_test import diebold_mariano
from mibel_forecasting.evaluation.metrics import mae, smape
from mibel_forecasting.features.technical_indicators import (
    TI_COLUMNS,
    compute_technical_indicators,
)
from mibel_forecasting.models.lear import LEAR

DIAG = Path('../reports/diagnostics').resolve()
DIAG.mkdir(parents=True, exist_ok=True)
CSV_PATH      = DIAG / 'technical_indicators_2026_05.csv'
MD_PATH       = DIAG / 'technical_indicators_2026_05.md'
COVERAGE_PATH = DIAG / 'technical_indicators_dropped_days_2026_05.csv'

PANEL_RANGE = {'panel_start': '2022-01-01', 'panel_end': '2024-12-31'}


## 1. Load panel preview

Loaded once in the main process for shape inspection. The parallel
workers reload it themselves and join the TI columns inside
``run_regime``.


In [ ]:
df = load_dam_panel(
    start=PANEL_RANGE['panel_start'], end=PANEL_RANGE['panel_end']
).dropna()
print(f'shape={df.shape}, tz={df.index.tz}')
print(f'range: {df.index.min()} -> {df.index.max()}')
df.head(2)


## 2. TI preview

Compute the eight TI columns on the full panel and inspect the
first post-warm-up rows. The displayed values must be finite for
every column except possibly ``ti_roc`` (zero-price hours; see audit
§"Indicator 5. ROC").


In [ ]:
ti = compute_technical_indicators(df)
print('ti shape:', ti.shape, 'columns:', list(ti.columns))
# Show rows around 2022-04-01 — past the Bollinger n=58 warm-up
ti.loc['2022-04-01':'2022-04-01 05:00'].round(4)


## 3. Regimes — same as notebook 03

For comparability, the regime windows and ``train_size`` choices
match notebook 03 exactly:

- 2022 H2 (crisis), ``train_size = 180D`` (pre-2022 cache empty);
- 2023 full year, ``train_size = 365D``;
- 2024 full year, ``train_size = 365D``;
- 2023-2024 pooled (derived from the two annual runs).

The 2022 caveats (crisis + Iberian exception gas cap) discussed in
notebook 03 §3 apply unchanged here.


In [ ]:
MODELS_LIST = list(MODEL_NAMES_WITH_TI)
ORDER = ['2022 (H2, crisis)', '2023 (full year)', '2024 (full year)', '2023-2024 pooled']

REGIME_SPECS = [
    {
        'regime':     '2022 (H2, crisis)',
        'start':      '2022-07-01',
        'end':        '2022-12-31',
        'train_size': '180D',
        'models':     MODELS_LIST,
        'with_ti':    True,
        **PANEL_RANGE,
    },
    {
        'regime':     '2023 (full year)',
        'start':      '2023-01-01',
        'end':        '2023-12-31',
        'train_size': '365D',
        'models':     MODELS_LIST,
        'with_ti':    True,
        **PANEL_RANGE,
    },
    {
        'regime':     '2024 (full year)',
        'start':      '2024-01-01',
        'end':        '2024-12-31',
        'train_size': '365D',
        'models':     MODELS_LIST,
        'with_ti':    True,
        **PANEL_RANGE,
    },
]
print(f'{len(MODELS_LIST)} models × {len(REGIME_SPECS)} regimes')


## 4. Smoke test — reproduce notebook 02 through ``run_regime``

Before the heavy parallel run, reproduce notebook 02's canonical
``naive MAE = 40.568`` and ``LEAR demand+wind rMAE = 0.394`` on the
same 2024-06-01..14 fortnight, through the same ``run_regime`` entry
point. This validates that the TI augmentation in ``run_regime``
does not inadvertently shift the non-TI variants away from nb02.

Tolerance: 3 % relative on both numbers.


In [ ]:
SMOKE_TOL_REL = 0.03

smoke_spec = {
    'regime':     'smoke (nb02 reference)',
    'start':      '2024-06-01',
    'end':        '2024-06-14',
    'train_size': '365D',
    'models':     ['naive', 'LEAR demand+wind'],
    'with_ti':    True,   # join TIs to panel even though models ignore them
    **PANEL_RANGE,
}

t0 = time.time()
smoke_result = run_regime(smoke_spec)
print(f'smoke run_regime done in {time.time()-t0:.1f}s')

smoke_metrics = {m['model']: m for m in smoke_result['metrics']}
smoke_naive_mae = smoke_metrics['naive']['MAE (EUR/MWh)']
smoke_dw_rmae   = smoke_metrics['LEAR demand+wind']['rMAE vs naive']

REF_NAIVE_MAE = 40.568
REF_DW_RMAE   = 0.394

def _rel(observed, expected):
    return abs(observed - expected) / expected if expected else float('inf')

dn = _rel(smoke_naive_mae, REF_NAIVE_MAE)
dr = _rel(smoke_dw_rmae,   REF_DW_RMAE)

print(f'naive MAE         {smoke_naive_mae:8.4f}  ref {REF_NAIVE_MAE:8.4f}  rel diff {dn*100:5.2f}%')
print(f'LEAR d+w rMAE     {smoke_dw_rmae:8.4f}  ref {REF_DW_RMAE:8.4f}  rel diff {dr*100:5.2f}%')

if dn > SMOKE_TOL_REL or dr > SMOKE_TOL_REL:
    raise RuntimeError(
        f'Smoke test failed: naive MAE rel diff {dn*100:.2f}% / '
        f'LEAR d+w rMAE rel diff {dr*100:.2f}% — tolerance was {SMOKE_TOL_REL*100:.0f}%. '
        'Halting before the full parallel run; investigate before proceeding.'
    )
print('SMOKE OK — proceeding to the parallel cross-regime run.')


## 5. Parallel backtests across regimes

Six models × three regimes. The TI variants are appreciably more
expensive than their non-TI counterparts (up to ≈ 511 features vs
247); the parallel layout brings total wall time down to roughly the
longest single-regime run.


In [ ]:
t0 = time.time()
n_workers = min(3, os.cpu_count() or 1)
print(f'launching {n_workers} workers for {len(REGIME_SPECS)} regimes...')

# ``cap_blas_threads`` lives in the package (importable by workers) and
# limits BLAS to 1 thread per worker process. Prevents the
# oversubscription / deadlock that stalled the first attempt at this
# notebook for >6h on Windows.
with ProcessPoolExecutor(
    max_workers=n_workers, initializer=cap_blas_threads
) as ex:
    results = list(ex.map(run_regime, REGIME_SPECS))

dt = time.time() - t0
print(f'all regimes done in {dt:.1f}s ({dt/60:.1f} min wall)')
for r in results:
    print(f'  {r["regime"]:22s}  ' + ' '.join(
        f'{m["model"]}={m["n_hours"]}h' for m in r['metrics']
    ))


## 6. Aggregate results — derive pooled regime


In [ ]:
forecasts: dict[tuple[str, str], pd.DataFrame] = {}
metric_rows: list[dict] = []
coverage_rows: list[dict] = []
for r in results:
    for model_name, f in r['forecasts'].items():
        forecasts[(r['regime'], model_name)] = f
    metric_rows.extend(r['metrics'])
    coverage_rows.extend(r['coverage'])

# Pooled 2023-2024 is derived; no extra LEAR fits.
pooled_label = '2023-2024 pooled'
for model_name in MODELS_LIST:
    pooled = pd.concat([
        forecasts[('2023 (full year)', model_name)],
        forecasts[('2024 (full year)', model_name)],
    ])
    forecasts[(pooled_label, model_name)] = pooled

naive_pool = forecasts[(pooled_label, 'naive')]
naive_pool_mae = mae(naive_pool['y_true'], naive_pool['y_pred'])
for model_name in MODELS_LIST:
    f = forecasts[(pooled_label, model_name)]
    m = mae(f['y_true'], f['y_pred'])
    s = smape(f['y_true'], f['y_pred'])
    r = m / naive_pool_mae if naive_pool_mae else float('nan')
    if model_name == 'naive':
        dm_stat, dm_pval, nw_lag = float('nan'), float('nan'), float('nan')
    else:
        dm = diebold_mariano(f['y_true'], naive_pool['y_pred'], f['y_pred'], horizon=24)
        dm_stat, dm_pval, nw_lag = dm.statistic, dm.p_value, dm.newey_west_lag
    metric_rows.append({
        'regime': pooled_label, 'model': model_name, 'n_hours': len(f),
        'MAE (EUR/MWh)': m, 'sMAPE (%)': s, 'rMAE vs naive': r,
        'DM stat vs naive': dm_stat, 'DM p-value vs naive': dm_pval, 'NW lag': nw_lag,
    })

for model_name in MODELS_LIST:
    f = forecasts[(pooled_label, model_name)]
    by_day = f['y_pred'].groupby(f.index.date).apply(lambda s: int(s.notna().sum()))
    coverage_rows.append({
        'regime': pooled_label, 'model': model_name,
        'days in panel':                  len(by_day),
        'full days predicted':            int((by_day == 24).sum()),
        'partial-hour days in panel':     int(((by_day > 0) & (by_day < 24)).sum()),
        'skipped (lag missing or NaN)':   int((by_day == 0).sum()),
    })

robust = pd.DataFrame(metric_rows)
coverage = pd.DataFrame(coverage_rows)
print(f'rows: metrics={len(robust)}, coverage={len(coverage)}')


## 7. Pairwise TI-vs-no-TI DM tests

Two pairwise comparisons per regime:

- `LEAR ar-only` vs `LEAR ar-only + TI` — isolates TI contribution
  without exogenous;
- `LEAR demand+wind` vs `LEAR demand+wind + TI` — isolates TI on top
  of Lago's canonical configuration.

Positive DM statistic ⇒ the no-TI variant has higher loss, i.e. the
TI variant wins. p-value below 0.05 ⇒ statistically significant.


In [ ]:
PAIRS = [
    ('LEAR ar-only', 'LEAR ar-only + TI'),
    ('LEAR demand+wind', 'LEAR demand+wind + TI'),
]

pair_rows = []
for regime in ORDER:
    for no_ti, ti in PAIRS:
        f_no_ti = forecasts[(regime, no_ti)]
        f_ti = forecasts[(regime, ti)]
        # Pair predictions on a common index; this matters for the TI
        # variants because a few days may be excluded by the TI NaN
        # filter (ROC p_{t-n}=0 cells) but present in the no-TI variant.
        merged = pd.concat({
            'y_true': f_no_ti['y_true'], 'y_no_ti': f_no_ti['y_pred'],
            'y_ti':   f_ti['y_pred'],
        }, axis=1).dropna()
        if len(merged) < 24 * 8:
            pair_rows.append({
                'regime': regime, 'no_ti_model': no_ti, 'ti_model': ti,
                'n_hours': len(merged),
                'MAE (no-TI)': float('nan'), 'MAE (TI)': float('nan'),
                'delta rMAE (TI - no-TI)': float('nan'),
                'DM stat (no-TI vs TI)': float('nan'),
                'DM p-value': float('nan'), 'NW lag': float('nan'),
            })
            continue
        mae_no_ti = mae(merged['y_true'], merged['y_no_ti'])
        mae_ti    = mae(merged['y_true'], merged['y_ti'])
        naive_f = forecasts[(regime, 'naive')]
        naive_pair = naive_f.reindex(merged.index).dropna()
        naive_mae_val = mae(naive_pair['y_true'], naive_pair['y_pred'])
        rmae_delta = (mae_ti - mae_no_ti) / naive_mae_val if naive_mae_val else float('nan')
        dm = diebold_mariano(merged['y_true'], merged['y_no_ti'], merged['y_ti'], horizon=24)
        pair_rows.append({
            'regime': regime, 'no_ti_model': no_ti, 'ti_model': ti,
            'n_hours': len(merged),
            'MAE (no-TI)': mae_no_ti, 'MAE (TI)': mae_ti,
            'delta rMAE (TI - no-TI)': rmae_delta,
            'DM stat (no-TI vs TI)': dm.statistic,
            'DM p-value': dm.p_value, 'NW lag': dm.newey_west_lag,
        })
pairwise = pd.DataFrame(pair_rows)
pairwise.round(4)


## 8. Coverage diagnostic


In [ ]:
coverage_pivot = coverage.pivot_table(
    index='regime', columns='model', values='full days predicted', aggfunc='first'
).reindex(ORDER)[MODELS_LIST]
coverage.to_csv(COVERAGE_PATH, index=False)
print(f'coverage CSV -> {COVERAGE_PATH}')
coverage_pivot


## 9. Robustness table

MAE, sMAPE, rMAE per (regime, model) plus DM-versus-naive for each
LEAR variant. The pairwise TI-vs-no-TI DM tests live in §7.


In [ ]:
robust['regime'] = pd.Categorical(robust['regime'], categories=ORDER, ordered=True)
robust['model']  = pd.Categorical(robust['model'],  categories=MODELS_LIST, ordered=True)
robust = robust.sort_values(['regime','model']).reset_index(drop=True)
robust.to_csv(CSV_PATH, index=False, float_format='%.4f')
print(f'metrics CSV -> {CSV_PATH}')
robust.round(4)


## 10. Lasso coefficient inspection — TI block

The audit anticipates that some Demir TIs may duplicate information
already in the AR price lags, in which case Lasso would zero them
out. Fit one representative LEAR ar-only + TI on the full 2024
training window and report what fraction of the TI block coefficients
were retained (i.e., non-zero) across the 24 hourly Lasso models.

A high zero-out rate is a finding, not a bug — it is the
identifiability counter-evidence the audit anticipated.


In [ ]:
# Build the train slice exactly as the rolling-forecast loop would
# at the first day of 2024-Q4 (representative cross-regime window).
train_end = pd.Timestamp('2024-09-30 23:00', tz='UTC')
train_start = train_end - pd.Timedelta('365D')
panel_with_ti = df.join(compute_technical_indicators(df))
train_slice = panel_with_ti.loc[train_start:train_end]

m_ti = LEAR(
    target_col='price_es',
    exogenous_cols=('es_demand_fc', 'es_wind_fc'),
    ti_cols=TI_COLUMNS,
).fit(train_slice)

# Feature layout: 96 price + 144 exog + 192 TI + 7 DoW = 439.
# Locate the TI block: between 96+144 and 96+144+192.
TI_START = 96 + 2 * 72
TI_END   = TI_START + 8 * 24

per_indicator = {}
for h in range(24):
    coefs = m_ti._lassos[h].coef_  # length = 439
    ti_block = coefs[TI_START:TI_END]   # 192 coefs per hour
    # Each TI occupies 24 contiguous cells (one per hour-of-day).
    for i, name in enumerate(TI_COLUMNS):
        start = i * 24
        end = start + 24
        block = ti_block[start:end]
        per_indicator.setdefault(name, []).extend(block.tolist())

coef_rows = []
for name, vals in per_indicator.items():
    arr = np.asarray(vals)
    coef_rows.append({
        'indicator': name,
        'n_coefs': int(len(arr)),
        '% non-zero': float((arr != 0).mean() * 100),
        'max |coef|': float(np.max(np.abs(arr))),
        'median |non-zero|': float(np.median(np.abs(arr[arr != 0]))) if (arr != 0).any() else float('nan'),
    })
coef_inspect = pd.DataFrame(coef_rows).round(4)
coef_inspect


## 11. Markdown export

Auto-generated narrative report at
`reports/diagnostics/technical_indicators_2026_05.md` covering:

- Validity preconditions inherited from nb03;
- Coverage diagnostic;
- Per-(regime, model) rMAE table;
- Pairwise TI-vs-no-TI DM tests;
- Coefficient inspection summary;
- Honest-reporting flags (small/negative TI gains; zeroed coefficients);
- Comparison to Demir 2019 (~4-5% RMSE reduction on Belgian DAM).


In [ ]:
LIMIT_SMALL_GAIN = 0.02  # |delta rMAE| below this counts as 'marginal'

# Honest-reporting checks
marginal_pairs = pairwise[pairwise['delta rMAE (TI - no-TI)'].abs() < LIMIT_SMALL_GAIN]
negative_pairs = pairwise[pairwise['delta rMAE (TI - no-TI)'] > 0]  # TI worse than no-TI
mass_zeroed = coef_inspect[coef_inspect['% non-zero'] < 20.0]

def _fmt(x, nd=4):
    if pd.isna(x): return ''
    return f'{x:.{nd}f}'

lines = [
    '# LEAR + Demir 2019 technical indicators (cross-regime)',
    '',
    '> _Auto-generated by `notebooks/04_lear_technical_indicators.ipynb`. Do not hand-edit._',
    '',
    '## Validity preconditions',
    '',
    '- All conventions from notebook 03 inherited: UTC panel (`b38b456`), target masking (`00a0aaf`), uniform partial-day skipping (`89cd2db` + `3d3a912`), LassoLarsIC noise-variance buffer (`685290b`).',
    '- TI parameters pinned by `reports/diagnostics/demir_2019_ti_parameter_audit_2026_05.md` (`0a81f1f` + `c52c3ed`). No MIBEL grid-search ablation run.',
    '- TI columns computed once on the full panel inside `run_regime` (see `src/mibel_forecasting/evaluation/_robustness.py`), then joined; non-TI variants ignore them via `ti_cols=()`.',
    '- 2022 carries `train_size=180D`; 2023 and 2024 use `train_size=365D`. Pooled 2023-2024 is derived from the two annual runs.',
    '',
    '## Coverage diagnostic',
    '',
    '`full days predicted` per (regime, model). Differences within a regime reflect the TI variants\' additional NaN filter (notably ROC `p_{t-n}=0` cells).',
    '',
    '| regime | model | days in panel | full days predicted | partial-hour days | skipped (lag/NaN) |',
    '| --- | --- | ---: | ---: | ---: | ---: |',
]
for _, r in coverage.iterrows():
    lines.append(
        f"| {r['regime']} | {r['model']} | {int(r['days in panel'])} "
        f"| {int(r['full days predicted'])} | {int(r['partial-hour days in panel'])} "
        f"| {int(r['skipped (lag missing or NaN)'])} |"
    )

lines += [
    '',
    '## Per-(regime, model) results',
    '',
    '| regime | model | n hours | MAE | sMAPE (%) | rMAE vs naive | DM stat | DM p-value | NW lag |',
    '| --- | --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |',
]
for _, r in robust.iterrows():
    lines.append(
        f"| {r['regime']} | {r['model']} | {int(r['n_hours'])} "
        f"| {_fmt(r['MAE (EUR/MWh)'], 3)} | {_fmt(r['sMAPE (%)'], 2)} | {_fmt(r['rMAE vs naive'])} "
        f"| {_fmt(r['DM stat vs naive'], 3)} | {_fmt(r['DM p-value vs naive'])} | {_fmt(r['NW lag'], 0)} |"
    )

lines += [
    '',
    '## Pairwise TI vs no-TI (Diebold-Mariano)',
    '',
    'DM stat sign convention: positive ⇒ no-TI loses (TI wins). Two pairs per regime.',
    '',
    '| regime | base | n hours | MAE (no-TI) | MAE (TI) | ΔrMAE (TI - no-TI) | DM stat | DM p-value |',
    '| --- | --- | ---: | ---: | ---: | ---: | ---: | ---: |',
]
for _, r in pairwise.iterrows():
    lines.append(
        f"| {r['regime']} | {r['no_ti_model']} -> {r['ti_model']} | {int(r['n_hours'])} "
        f"| {_fmt(r['MAE (no-TI)'], 3)} | {_fmt(r['MAE (TI)'], 3)} "
        f"| {_fmt(r['delta rMAE (TI - no-TI)'])} | {_fmt(r['DM stat (no-TI vs TI)'], 3)} | {_fmt(r['DM p-value'])} |"
    )

lines += [
    '',
    '## Coefficient inspection (LEAR ar-only + TI on 2024 trailing year)',
    '',
    'Fraction of the TI block coefficients that survived Lasso shrinkage, per indicator. Low retention (< 20 %) is flagged: it is consistent with the audit\'s expectation that some Demir indicators duplicate information already in the price-lag block.',
    '',
    '| indicator | n coefs | % non-zero | max \|coef\| | median \|non-zero\| |',
    '| --- | ---: | ---: | ---: | ---: |',
]
for _, r in coef_inspect.iterrows():
    lines.append(
        f"| {r['indicator']} | {int(r['n_coefs'])} | "
        f"{_fmt(r['% non-zero'], 2)} | {_fmt(r['max |coef|'])} | {_fmt(r['median |non-zero|'])} |"
    )

lines += ['', '## Honest reporting', '']
if not marginal_pairs.empty:
    lines += ['**Marginal TI gain (|ΔrMAE| < 0.02)** — TI did not move rMAE meaningfully in:', '']
    for _, p in marginal_pairs.iterrows():
        lines.append(
            f"- {p['regime']} / {p['no_ti_model']} -> {p['ti_model']}: "
            f"ΔrMAE = {p['delta rMAE (TI - no-TI)']:+.4f}, DM p = {p['DM p-value']:.4f}"
        )
    lines.append('')

if not negative_pairs.empty:
    lines += ['**Negative TI contribution (TI worse than no-TI):**', '']
    for _, p in negative_pairs.iterrows():
        lines.append(
            f"- {p['regime']} / {p['no_ti_model']} -> {p['ti_model']}: "
            f"ΔrMAE = {p['delta rMAE (TI - no-TI)']:+.4f}, DM p = {p['DM p-value']:.4f}"
        )
    lines.append('')

if marginal_pairs.empty and negative_pairs.empty:
    lines += [
        'Every (regime, pair) shows a meaningful (|ΔrMAE| >= 0.02) TI improvement over the no-TI baseline.',
        '',
    ]

if not mass_zeroed.empty:
    lines += [
        '**Largely zeroed TI block — the Lasso retained < 20 % of the coefficients for:**',
        '',
    ]
    for _, r in mass_zeroed.iterrows():
        lines.append(f"- `{r['indicator']}`: {r['% non-zero']:.1f} % non-zero coefficients retained")
    lines += [
        '',
        "This is *expected* behaviour per the audit: these indicators duplicate information already in the AR price-lag block. The corresponding rows in the per-indicator table above with very low '% non-zero' should not be read as 'Lasso bugs' — they are the identifiability counter-evidence the audit anticipated.",
        '',
    ]

lines += [
    '## Comparison to Demir et al. (2019)',
    '',
    'Demir reported a peak RMSE reduction of **4.50 %** for Huber Regression on Belgian DAM with the best TI per model. Our pairwise %RMSE reductions on MIBEL (computed as `100 * (MAE_no-TI - MAE_TI) / MAE_no-TI` for the cleanest analogue) per regime are:',
    '',
    '| regime | base | %MAE reduction |',
    '| --- | --- | ---: |',
]
for _, p in pairwise.iterrows():
    if pd.isna(p['MAE (no-TI)']) or pd.isna(p['MAE (TI)']):
        rel = ''
    else:
        rel = f"{(p['MAE (no-TI)'] - p['MAE (TI)']) / p['MAE (no-TI)'] * 100:+.2f}%"
    lines.append(f"| {p['regime']} | {p['no_ti_model']} -> {p['ti_model']} | {rel} |")

lines += [
    '',
    'Hypotheses for any direction or magnitude mismatch vs Demir on Belgian DAM:',
    '',
    '1. **Higher solar penetration in MIBEL 2023-2024** — mid-day prices collapse to zero in a regime-dependent way. Demir\'s ROC `p_{t-n}=0` edge case (audit §"Indicator 5. ROC") fires far more often on MIBEL than on Belgian DAM, partially zeroing the ROC contribution.',
    '2. **Different price dynamics** — Iberian DAM exhibits longer autoregressive horizons than Belgian DAM (cross-border interconnection capacity, gas-cap regime), which the LEAR 96-feature AR block already captures; TI features may therefore add proportionally less new information.',
    '3. **No MIBEL grid-search** — Demir grid-searched all hyperparameters per model on Belgian DAM. We adopted Demir\'s linear-model optima (EMA s=2, %B n=58, MACD (2,26,9)) and cross-model defaults for MOM/ROC/Coppock. A MIBEL-specific grid-search is listed as future work in the audit and could shift the magnitude.',
    '',
    '## Provenance',
    '',
    '- Notebook: `notebooks/04_lear_technical_indicators.ipynb`',
    '- Underlying metrics: `reports/diagnostics/technical_indicators_2026_05.csv`',
    '- Dropped-day diagnostic: `reports/diagnostics/technical_indicators_dropped_days_2026_05.csv`',
    f'- Date generated: {pd.Timestamp.now().strftime("%Y-%m-%d")}',
    '',
]

MD_PATH.write_text('\n'.join(lines), encoding='utf-8')
print(f'markdown -> {MD_PATH}')
print()
print('\n'.join(lines[:30]))


## 12. Reading the result

The honest-reporting clauses above will fire if:

- TI does not materially help (`|ΔrMAE| < 0.02`) — flagged per
  (regime, base);
- TI makes things worse (`ΔrMAE > 0`) — flagged with the same
  granularity;
- Lasso shrinks more than 80 % of the coefficients on any indicator
  — flagged in the coefficient inspection.

A clean win across all regimes and both base configurations is the
expected best case; a mixed picture (small gains here, marginal
losses there) is the most likely outcome on MIBEL given the
high-solar regime and the strong AR autocorrelation of the price.

**Next** — sub-objective 3.2 will introduce the DNN backbone
(Lago 2021 §3.3) and revisit the TI question on a non-linear
estimator.
